## Classe `CaptureManager`
O que representa esta classe? (responda abaixo)


### Método `CaptureManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core):
        super().__init__()
        self.core = core
        self.current_image: Optional[np.ndarray] = None
        self.raw_image: Optional[np.ndarray] = None
        self.crop_enabled: bool = False
        self.crop_bbox: Optional[tuple] = None
        self._inspect_after_capture: Optional[str] = None

### Método `CaptureManager.set_crop_settings`
O que faz este método? (responda abaixo)


In [ ]:
def set_crop_settings(self, enabled: bool, bbox: Optional[tuple]=None):
        self.crop_enabled = enabled
        self.crop_bbox = bbox

### Método `CaptureManager.reset_crop`
O que faz este método? (responda abaixo)


In [ ]:
def reset_crop(self):
        self.crop_enabled = False
        self.crop_bbox = None

### Método `CaptureManager.set_inspect_after_capture`
O que faz este método? (responda abaixo)


In [ ]:
def set_inspect_after_capture(self, inspection_type: Optional[str]=None):
        self._inspect_after_capture = inspection_type

### Método `CaptureManager.get_current_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_current_image(self) -> Optional[np.ndarray]:
        return self.current_image

### Método `CaptureManager.get_raw_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_raw_image(self) -> Optional[np.ndarray]:
        return self.raw_image

### Método `CaptureManager.process_captured_image`
O que faz este método? (responda abaixo)


In [ ]:
def process_captured_image(self, capture_result: Dict[str, Any]) -> Dict[str, Any]:
        try:
            raw_image = capture_result.get('image')
            camera_info = capture_result.get('camera_info', {})
            timestamp = capture_result.get('timestamp', '')
            if raw_image is None:
                raise Exception('Nenhuma imagem no resultado de captura')
            self.raw_image = raw_image
            processed_image = raw_image
            if self.crop_enabled and self.crop_bbox:
                try:
                    ops = [{'name': 'crop', 'bbox': self.crop_bbox}]
                    processed_image = self.core.preprocess_image(raw_image, ops)
                except Exception as e:
                    raise Exception(f'Falha ao aplicar crop: {e}')
            self.current_image = processed_image
            return {'image': processed_image, 'raw_image': raw_image, 'camera_info': camera_info, 'timestamp': timestamp, 'shape': processed_image.shape}
        except Exception as e:
            self.error_occurred.emit(str(e))
            return None

### Método `CaptureManager.get_image_for_display`
O que faz este método? (responda abaixo)


In [ ]:
def get_image_for_display(self, image: Optional[np.ndarray]=None) -> Optional[np.ndarray]:
        if image is None:
            image = self.current_image
        if image is None:
            return None
        if len(image.shape) == 3 and image.shape[2] == 3:
            try:
                return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            except:
                return image
        return image

### Método `CaptureManager.get_image_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_image_info(self) -> str:
        if self.current_image is None:
            return 'Nenhuma imagem capturada'
        shape = self.current_image.shape
        dtype = self.current_image.dtype
        size_mb = self.current_image.nbytes / (1024 * 1024)
        return f'Shape: {shape} | Dtype: {dtype} | Size: {size_mb:.2f}MB'

### Método `CaptureManager.clear_images`
O que faz este método? (responda abaixo)


In [ ]:
def clear_images(self):
        self.current_image = None
        self.raw_image = None
        self._inspect_after_capture = None

### Método `CaptureManager.should_inspect_after_capture`
O que faz este método? (responda abaixo)


In [ ]:
def should_inspect_after_capture(self) -> Optional[str]:
        return self._inspect_after_capture

## Classe `CaptureThread`
O que representa esta classe? (responda abaixo)


### Método `CaptureThread.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core):
        super().__init__()
        self.core = core
        self._should_capture = False

### Método `CaptureThread.capture`
O que faz este método? (responda abaixo)


In [ ]:
def capture(self):
        self._should_capture = True
        self.start()

### Método `CaptureThread.run`
O que faz este método? (responda abaixo)


In [ ]:
def run(self):
        try:
            if not self._should_capture:
                return
            result = self.core.capture_image()
            if result:
                self.image_captured.emit(result)
            else:
                self.error_occurred.emit('Falha ao capturar imagem')
        except Exception as e:
            self.error_occurred.emit(f'Erro na captura: {str(e)}')
        finally:
            self._should_capture = False

## Classe `CaptureManagerQt`
O que representa esta classe? (responda abaixo)


### Método `CaptureManagerQt.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core, camera_profiles=None):
        super().__init__()
        self.manager = CaptureManager(core, camera_profiles)
        core.register_callback('capture_completed', self._on_capture_completed)
        core.register_callback('capture_error', self._on_capture_error)
        self.capture_thread = CaptureThread(core)
        self.capture_thread.image_captured.connect(self._on_image_captured_raw)
        self.capture_thread.error_occurred.connect(self._on_capture_error)
        self.core = core
        self._log('info', 'Manager Qt criado')

### Método `CaptureManagerQt._log`
O que faz este método? (responda abaixo)


In [ ]:
def _log(self, level: str, msg: str):
        try:
            logger_method = getattr(log, level, None)
            if callable(logger_method):
                logger_method(f'[CaptureManagerQt] {msg}')
            else:
                log.info(f'[CaptureManagerQt] {msg}')
        except Exception:
            log.info(f'[CaptureManagerQt] {msg}')

### Método `CaptureManagerQt.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        return self.manager.initialize()

### Método `CaptureManagerQt.cleanup`
O que faz este método? (responda abaixo)


In [ ]:
def cleanup(self) -> None:
        if self.capture_thread.isRunning():
            self.capture_thread.wait()
        self.manager.cleanup()

### Método `CaptureManagerQt.set_crop_settings`
O que faz este método? (responda abaixo)


In [ ]:
def set_crop_settings(self, enabled: bool, bbox: Optional[tuple]=None):
        self.manager.set_crop_settings(enabled, bbox)
        self._notify_crop_change()

### Método `CaptureManagerQt.reset_crop`
O que faz este método? (responda abaixo)


In [ ]:
def reset_crop(self):
        self.manager.reset_crop()
        self._notify_crop_change()

### Método `CaptureManagerQt.set_inspect_after_capture`
O que faz este método? (responda abaixo)


In [ ]:
def set_inspect_after_capture(self, inspection_type: Optional[str]=None):
        self.manager.set_inspect_after_capture(inspection_type)

### Método `CaptureManagerQt.should_inspect_after_capture`
O que faz este método? (responda abaixo)


In [ ]:
def should_inspect_after_capture(self) -> Optional[str]:
        return self.manager.should_inspect_after_capture()

### Método `CaptureManagerQt.clear_images`
O que faz este método? (responda abaixo)


In [ ]:
def clear_images(self):
        self.manager.clear_images()

### Método `CaptureManagerQt.apply_profile`
O que faz este método? (responda abaixo)


In [ ]:
def apply_profile(self, profile_name: str) -> bool:
        return self.manager.apply_profile(profile_name)

### Método `CaptureManagerQt.capture_image`
O que faz este método? (responda abaixo)


In [ ]:
def capture_image(self):
        self.capture_started.emit()
        self.capture_thread.capture()

### Método `CaptureManagerQt._on_image_captured_raw`
O que faz este método? (responda abaixo)


In [ ]:
def _on_image_captured_raw(self, capture_result: Dict[str, Any]):
        try:
            result = self.manager.process_captured_image(capture_result)
            if result:
                self.image_captured.emit(result)
            else:
                self.error_occurred.emit('Falha ao processar imagem')
        except Exception as e:
            self._log('error', f'Erro ao processar: {e}')
            self.error_occurred.emit(str(e))
        finally:
            self.capture_finished.emit()

### Método `CaptureManagerQt._on_capture_completed`
O que faz este método? (responda abaixo)


In [ ]:
def _on_capture_completed(self, data: Dict[str, Any]):
        self.capture_completed.emit(data)

### Método `CaptureManagerQt._on_capture_error`
O que faz este método? (responda abaixo)


In [ ]:
def _on_capture_error(self, data: Dict[str, Any]):
        error = data.get('error', 'Erro desconhecido')
        self.error_occurred.emit(error)

### Método `CaptureManagerQt.get_image_for_display`
O que faz este método? (responda abaixo)


In [ ]:
def get_image_for_display(self) -> Optional[np.ndarray]:
        image = self.manager.get_current_image()
        if image is None:
            return None
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return rgb_image

### Método `CaptureManagerQt.get_image_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_image_info(self) -> Dict[str, Any]:
        return self.manager.get_image_info()

### Método `CaptureManagerQt.get_current_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_current_image(self) -> Optional[np.ndarray]:
        return self.manager.get_current_image()

### Método `CaptureManagerQt.get_raw_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_raw_image(self) -> Optional[np.ndarray]:
        return self.manager.get_raw_image()

### Método `CaptureManagerQt._notify_crop_change`
O que faz este método? (responda abaixo)


In [ ]:
def _notify_crop_change(self):
        info = self.get_image_info()
        self.capture_completed.emit({'crop_enabled': self.manager.crop_enabled, 'crop_bbox': self.manager.crop_bbox, 'info': info})

### Método `CaptureManagerQt.is_crop_enabled`
O que faz este método? (responda abaixo)


In [ ]:
def is_crop_enabled(self) -> bool:
        return self.manager.crop_enabled

### Método `CaptureManagerQt.get_crop_bbox`
O que faz este método? (responda abaixo)


In [ ]:
def get_crop_bbox(self) -> Optional[tuple]:
        return self.manager.crop_bbox

### Método `CaptureManagerQt.current_image`
O que faz este método? (responda abaixo)


In [ ]:
def current_image(self) -> Optional[np.ndarray]:
        return self.manager.get_current_image()

### Método `CaptureManagerQt.raw_image`
O que faz este método? (responda abaixo)


In [ ]:
def raw_image(self) -> Optional[np.ndarray]:
        return self.manager.get_raw_image()

## Classe `HistoryManager`
O que representa esta classe? (responda abaixo)


### Método `HistoryManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core, results_dir: Optional[Path]=None):
        self.core = core
        self.results_dir = Path(results_dir) if results_dir else Path('data/results')
        self.loaded_history: List[Dict] = []

### Método `HistoryManager.load_history`
O que faz este método? (responda abaixo)


In [ ]:
def load_history(self) -> List[Dict]:
        try:
            if not self.results_dir.exists():
                return []
            history = []
            for result_dir in sorted(self.results_dir.iterdir(), reverse=True):
                if not result_dir.is_dir():
                    continue
                inspection_file = result_dir / 'inspection_data.json'
                if inspection_file.exists():
                    try:
                        with open(inspection_file, 'r') as f:
                            data = json.load(f)
                        history.append({'timestamp': data.get('timestamp'), 'inspection_type': data.get('inspection_type'), 'model_name': data.get('model_name'), 'path': str(result_dir), 'data': data.get('results', {})})
                    except Exception as e:
                        print(f'Falha ao carregar {inspection_file}: {e}')
            self.loaded_history = history
            return history
        except Exception as e:
            print(f'Erro ao carregar histórico: {e}')
            return []

### Método `HistoryManager.get_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_item(self, index: int) -> Optional[Dict]:
        if 0 <= index < len(self.loaded_history):
            return self.loaded_history[index]
        return None

### Método `HistoryManager.get_history_by_timestamp`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_by_timestamp(self, timestamp: str) -> Optional[Dict]:
        for item in self.loaded_history:
            if item['timestamp'] == timestamp:
                return item
        return None

### Método `HistoryManager.get_history_count`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_count(self) -> int:
        return len(self.loaded_history)

### Método `HistoryManager.view_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def view_history_item(self, item_index: int) -> Optional[str]:
        item = self.get_history_item(item_index)
        if not item:
            return None
        lines = []
        lines.append(f"{'=' * 60}")
        lines.append(f"Data/Hora: {item.get('timestamp', 'N/A')}")
        lines.append(f"Tipo: {item.get('inspection_type', 'N/A').upper()}")
        lines.append(f"Modelo: {item.get('model_name', 'N/A')}")
        lines.append(f"{'=' * 60}")
        results = item.get('data', {})
        if item.get('inspection_type') == 'segmentation':
            lines.append(f"Classe: {results.get('class', 'N/A')}")
            lines.append(f"Confiança: {results.get('confidence', 0):.2%}")
            lines.append(f"Área: {results.get('area', 'N/A')}")
        elif item.get('inspection_type') == 'classification':
            lines.append(f"Classe Predita: {results.get('predicted_class', 'N/A')}")
            lines.append(f"Confiança: {results.get('confidence', 0):.2%}")
            if 'class_confidences' in results:
                lines.append('\nConfiança por classe:')
                for cls, conf in results['class_confidences'].items():
                    lines.append(f'  - {cls}: {conf:.2%}')
        lines.append(f"{'=' * 60}")
        return '\n'.join(lines)

### Método `HistoryManager.export_to_csv`
O que faz este método? (responda abaixo)


In [ ]:
def export_to_csv(self, output_path: Path) -> bool:
        try:
            if not self.loaded_history:
                return False
            with open(output_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Tipo', 'Modelo', 'Resultado', 'Confiança'])
                for item in self.loaded_history:
                    timestamp = item.get('timestamp', '')
                    inspection_type = item.get('inspection_type', '')
                    model_name = item.get('model_name', '')
                    data = item.get('data', {})
                    if inspection_type == 'segmentation':
                        resultado = data.get('class', 'N/A')
                    else:
                        resultado = data.get('predicted_class', 'N/A')
                    confianca = f"{data.get('confidence', 0):.2%}"
                    writer.writerow([timestamp, inspection_type, model_name, resultado, confianca])
            return True
        except Exception as e:
            print(f'Erro ao exportar CSV: {e}')
            return False

### Método `HistoryManager.delete_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def delete_history_item(self, timestamp: str) -> bool:
        try:
            result_dir = self.results_dir / timestamp
            if result_dir.exists():
                import shutil
                shutil.rmtree(result_dir)
                self.loaded_history = [item for item in self.loaded_history if item['timestamp'] != timestamp]
                return True
            return False
        except Exception as e:
            print(f'Erro ao deletar histórico: {e}')
            return False

### Método `HistoryManager.clear_all_history`
O que faz este método? (responda abaixo)


In [ ]:
def clear_all_history(self) -> bool:
        try:
            if self.results_dir.exists():
                import shutil
                shutil.rmtree(self.results_dir)
                self.results_dir.mkdir(parents=True, exist_ok=True)
                self.loaded_history = []
                return True
            return False
        except Exception as e:
            print(f'Erro ao limpar histórico: {e}')
            return False

### Método `HistoryManager.get_summary_stats`
O que faz este método? (responda abaixo)


In [ ]:
def get_summary_stats(self) -> Dict[str, Any]:
        if not self.loaded_history:
            return {'total': 0}
        segmentation_count = sum((1 for item in self.loaded_history if item.get('inspection_type') == 'segmentation'))
        classification_count = sum((1 for item in self.loaded_history if item.get('inspection_type') == 'classification'))
        models = {}
        for item in self.loaded_history:
            model = item.get('model_name', 'Unknown')
            models[model] = models.get(model, 0) + 1
        return {'total': len(self.loaded_history), 'segmentation': segmentation_count, 'classification': classification_count, 'models': models}

## Classe `HistoryManagerQt`
O que representa esta classe? (responda abaixo)


### Método `HistoryManagerQt.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core, results_dir: Optional[Path]=None):
        super().__init__()
        self.manager = CoreHistoryManager(core, results_dir)
        if hasattr(core, 'register_callback'):
            core.register_callback('history_loaded', self._on_history_loaded)
            core.register_callback('history_modified', self._on_history_modified)
            core.register_callback('history_error', self._on_history_error)

### Método `HistoryManagerQt.load_history`
O que faz este método? (responda abaixo)


In [ ]:
def load_history(self):
        try:
            result = self.manager.load_history()
            self.history_loaded.emit()
            return result
        except Exception as e:
            self.error_occurred.emit(str(e))
            return []

### Método `HistoryManagerQt.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        try:
            return self.manager.initialize()
        except Exception as e:
            self.error_occurred.emit(str(e))
            return False

### Método `HistoryManagerQt.cleanup`
O que faz este método? (responda abaixo)


In [ ]:
def cleanup(self) -> None:
        self.manager.cleanup()

### Método `HistoryManagerQt.get_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_item(self, index: int) -> Optional[Dict]:
        return self.manager.get_history_item(index)

### Método `HistoryManagerQt.get_history_by_timestamp`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_by_timestamp(self, timestamp: str) -> Optional[Dict]:
        return self.manager.get_history_by_timestamp(timestamp)

### Método `HistoryManagerQt.get_history_count`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_count(self) -> int:
        return self.manager.get_history_count()

### Método `HistoryManagerQt.view_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def view_history_item(self, item_index: int) -> Optional[str]:
        return self.manager.view_history_item(item_index)

### Método `HistoryManagerQt.export_to_csv`
O que faz este método? (responda abaixo)


In [ ]:
def export_to_csv(self, output_path: Path) -> bool:
        try:
            success = self.manager.export_to_csv(output_path)
            if success:
                self.history_changed.emit()
            return success
        except Exception as e:
            self.error_occurred.emit(str(e))
            return False

### Método `HistoryManagerQt.delete_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def delete_history_item(self, timestamp: str) -> bool:
        try:
            success = self.manager.delete_history_item(timestamp)
            if success:
                self.history_changed.emit()
            return success
        except Exception as e:
            self.error_occurred.emit(str(e))
            return False

### Método `HistoryManagerQt.clear_all_history`
O que faz este método? (responda abaixo)


In [ ]:
def clear_all_history(self) -> bool:
        try:
            success = self.manager.clear_all_history()
            if success:
                self.history_changed.emit()
            return success
        except Exception as e:
            self.error_occurred.emit(str(e))
            return False

### Método `HistoryManagerQt.get_summary_stats`
O que faz este método? (responda abaixo)


In [ ]:
def get_summary_stats(self) -> Dict[str, Any]:
        return self.manager.get_summary_stats()

### Método `HistoryManagerQt._on_history_loaded`
O que faz este método? (responda abaixo)


In [ ]:
def _on_history_loaded(self):
        self.history_loaded.emit()

### Método `HistoryManagerQt._on_history_modified`
O que faz este método? (responda abaixo)


In [ ]:
def _on_history_modified(self):
        self.history_changed.emit()

### Método `HistoryManagerQt._on_history_error`
O que faz este método? (responda abaixo)


In [ ]:
def _on_history_error(self, error: str):
        self.error_occurred.emit(error)

## Classe `InspectionManager`
O que representa esta classe? (responda abaixo)


### Método `InspectionManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core):
        super().__init__()
        self.core = core
        self.current_results: Optional[Dict] = None
        self.inspection_image: Optional[np.ndarray] = None
        self.last_inspection_type: Optional[str] = None
        self.last_model_name: Optional[str] = None

### Método `InspectionManager.perform_inspection`
O que faz este método? (responda abaixo)


In [ ]:
def perform_inspection(self, image: np.ndarray, inspection_type: str, model_name: str, params: Optional[Dict[str, Any]]=None) -> Optional[Dict]:
        try:
            if image is None:
                raise ValueError('Imagem é None')
            self.inspection_started.emit()
            self.inspection_image = image
            self.last_inspection_type = inspection_type
            self.last_model_name = model_name
            results = self.core.run_model(image=image, model_name=model_name, inspection_type=inspection_type, confidence=params.get('confidence', 0.5) if params else 0.5)
            self.current_results = results
            self.inspection_completed.emit(results, inspection_type)
            return results
        except Exception as e:
            self.error_occurred.emit(str(e))
            return None

### Método `InspectionManager.get_current_results`
O que faz este método? (responda abaixo)


In [ ]:
def get_current_results(self) -> Optional[Dict]:
        return self.current_results

### Método `InspectionManager.format_results_text`
O que faz este método? (responda abaixo)


In [ ]:
def format_results_text(self, results: Dict, inspection_type: str) -> str:
        if not results:
            return 'Sem resultados'
        lines = []
        lines.append(f"{'=' * 60}")
        lines.append(f'Tipo: {inspection_type.upper()}')
        lines.append(f"Modelo: {results.get('model_name', 'N/A')}")
        lines.append(f"Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        lines.append(f"{'=' * 60}")
        if inspection_type == 'segmentation':
            lines.append(f"Classe: {results.get('class', 'N/A')}")
            lines.append(f"Confiança: {results.get('confidence', 0):.2%}")
            lines.append(f"Área segmentada: {results.get('area', 'N/A')}")
        elif inspection_type == 'classification':
            lines.append(f"Classe Predita: {results.get('predicted_class', 'N/A')}")
            lines.append(f"Confiança: {results.get('confidence', 0):.2%}")
            if 'class_confidences' in results:
                lines.append('\nConfiança por classe:')
                for cls, conf in results['class_confidences'].items():
                    lines.append(f'  - {cls}: {conf:.2%}')
        lines.append(f"{'=' * 60}")
        return '\n'.join(lines)

### Método `InspectionManager.get_results_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_results_image(self) -> Optional[np.ndarray]:
        if self.current_results is None:
            return self.inspection_image
        if 'annotated_image' in self.current_results:
            return self.current_results['annotated_image']
        return self.inspection_image

### Método `InspectionManager.save_results`
O que faz este método? (responda abaixo)


In [ ]:
def save_results(self, image: Optional[np.ndarray]=None, output_dir: Optional[Path]=None) -> Optional[Path]:
        try:
            if self.current_results is None:
                raise ValueError('Sem resultados para salvar')
            if image is None:
                image = self.inspection_image
            if output_dir is None:
                output_dir = Path('data/results')
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            result_dir = output_dir / timestamp
            result_dir.mkdir(parents=True, exist_ok=True)
            data = {'timestamp': timestamp, 'inspection_type': self.last_inspection_type, 'model_name': self.last_model_name, 'results': self.current_results}
            with open(result_dir / 'inspection_data.json', 'w') as f:
                json.dump(data, f, indent=2, default=str)
            if image is not None:
                import cv2
                cv2.imwrite(str(result_dir / 'inspection_image.png'), image)
            return result_dir
        except Exception as e:
            self.error_occurred.emit(f'Falha ao salvar resultados: {e}')
            return None

### Método `InspectionManager.clear_results`
O que faz este método? (responda abaixo)


In [ ]:
def clear_results(self):
        self.current_results = None
        self.inspection_image = None
        self.last_inspection_type = None
        self.last_model_name = None

## 🧹 LIMPEZA DE MEMÓRIA

In [ ]:
import gc

# Limpar cache e liberar memória
try:
    del model
    print("✓ Modelo deletado")
except NameError:
    print("ℹ Nenhum modelo em memória")

gc.collect()
print("✓ Garbage collection executado")